# Ranking metrics from scratch

Issue: #13

## Question

Can we trust our NDCG@K, recall@K and OTTO weighted recall@20? We write them by hand and check them against scikit-learn.

## Data used

No OTTO data. Toy cases with hand-calculated answers, plus random cases compared with scikit-learn.

Code: `notebooks/metrics.py` (import path `from metrics import ...`). It stays out of `src/ranking/` until issue #52.

In [ ]:
import math
import random
import sys

import numpy as np
from sklearn.metrics import ndcg_score

sys.path.insert(0, ".")  # notebooks are run from the notebooks/ folder or the repo root
sys.path.insert(0, "notebooks")
from metrics import (
    item_gains,
    ndcg_at_k,
    otto_recall_by_type,
    otto_weighted_recall,
    otto_weighted_recall_session,
    recall_at_k,
)

L2 = math.log2

## What each metric rewards

- **NDCG@K** rewards putting *more valuable* items (order 3, cart 2, click 1) *higher*. A relevant item at position 1 counts fully, at position 3 it counts 1/log2(4) = 0.5. It is divided by the best possible ordering, so 1.0 means perfect.
- **recall@K** rewards *finding* the relevant items at all in the top K, whatever their order.
- **OTTO weighted recall@20** rewards finding what the user later did, with orders counting six times more than clicks (0.10, 0.30, 0.60). Per type it divides by min(20, number of items), so a user with 50 clicks is not expected to have all 50 found.

## Toy cases with hand-calculated answers

In [ ]:
# Each expected value is written out by hand from the definition (linear gain, log2(rank + 1) discount).
cases = []

# 1. Only relevant item ranked first: dcg = 1/log2(2) = 1, ideal = 1
cases.append(([1, 2, 3], {"click": {1}}, 1.0))
# 2. Same item ranked second: dcg = 1/log2(3), ideal = 1
cases.append(([2, 1, 3], {"click": {1}}, 1 / L2(3)))
# 3. Click item 1 (gain 1) and order item 5 (gain 3). Ranking [1, 5]
#    dcg = 1/log2(2) + 3/log2(3); ideal = 3/log2(2) + 1/log2(3)
cases.append(([1, 5], {"click": {1}, "order": {5}}, (1 + 3 / L2(3)) / (3 + 1 / L2(3))))
# 4. Repeated item in the ranking counts once: [1, 1, 5] behaves like [1, 5]
cases.append(([1, 1, 5], {"click": {1}, "order": {5}}, (1 + 3 / L2(3)) / (3 + 1 / L2(3))))
# 5. Item with click and cart takes the higher gain (2). Ranking [7]: dcg = 2, ideal = 2
cases.append(([7], {"click": {7}, "cart": {7}}, 1.0))
# 6. Relevant item only at position 11: outside the top 10
cases.append((list(range(100, 110)) + [1], {"click": {1}}, 0.0))
# 7. No labels at all: defined as 0.0
cases.append(([1, 2, 3], {"click": set(), "cart": set(), "order": set()}, 0.0))
# 8. Nothing found: 0.0
cases.append(([8, 9], {"click": {1}}, 0.0))

for ranking, labels, expected in cases:
    got = ndcg_at_k(ranking, labels, k=10)
    assert abs(got - expected) < 1e-9, (ranking, labels, got, expected)
print(len(cases), "NDCG toy cases pass")

In [ ]:
# recall@K by hand
assert recall_at_k([1, 9, 2], {1, 2, 3, 4}, k=3) == 2 / 4          # found 1 and 2 of 4
assert recall_at_k([1, 9, 2], {1, 2, 3, 4}, k=1) == 1 / 4
assert recall_at_k([1, 1, 1], {1}, k=3) == 1.0                     # duplicates
assert recall_at_k([1, 2], set(), k=3) == 0.0                      # nothing relevant
print("recall@K toy cases pass")

# OTTO weighted recall by hand
labels = {"click": {1, 2}, "cart": {3}, "order": {4}}
ranking = [1, 3, 9, 4]
by_type = otto_recall_by_type(ranking, labels, k=20)
assert by_type == {"click": 1 / 2, "cart": 1.0, "order": 1.0}, by_type
assert abs(otto_weighted_recall([by_type]) - (0.10 * 0.5 + 0.30 * 1 + 0.60 * 1)) < 1e-12

# min(20, n): 50 clicked items, top 20 all hits -> recall 1.0, not 20/50
labels50 = {"click": set(range(50))}
assert otto_recall_by_type(list(range(20)), labels50, k=20) == {"click": 1.0}

# two sessions: click means are taken only over sessions that have clicks
s1 = {"click": 1.0}
s2 = {"click": 0.0, "order": 1.0}
assert abs(otto_weighted_recall([s1, s2]) - (0.10 * 0.5 + 0.60 * 1.0)) < 1e-12

# single-session version renormalises over the types present
assert abs(otto_weighted_recall_session([1], {"click": {1}}, k=20) - 1.0) < 1e-12
print("OTTO recall toy cases pass")

## Check NDCG against scikit-learn on random cases

scikit-learn needs a score for every item. Ranked items get descending scores, and items outside the ranking get lower distinct scores, so there are no ties. The ranking is at least k long so the top k is the same on both sides.

In [ ]:
rng = random.Random(0)
GAINS = {"click": 1, "cart": 2, "order": 3}
worst = 0.0
n_cases = 500
for _ in range(n_cases):
    universe = list(range(60))
    ranking = rng.sample(universe, rng.randint(10, 30))
    labels = {t: set(rng.sample(universe, rng.randint(0, 6))) for t in GAINS}
    gains = item_gains(labels, GAINS)
    if not gains:
        continue
    k = rng.choice([1, 3, 5, 10])
    y_true = np.array([[gains.get(i, 0) for i in universe]], dtype=float)
    scores = {item: 1000 - pos for pos, item in enumerate(ranking)}
    rest = [i for i in universe if i not in scores]
    scores.update({item: -(n + 1) for n, item in enumerate(rest)})
    y_score = np.array([[scores[i] for i in universe]], dtype=float)
    expected = ndcg_score(y_true, y_score, k=k)
    got = ndcg_at_k(ranking, labels, k=k)
    worst = max(worst, abs(got - expected))
    assert abs(got - expected) < 1e-6, (ranking, labels, k, got, expected)
print(f"{n_cases} random cases match scikit-learn, worst difference {worst:.2e}")

## Result

All toy cases match hand calculations to 1e-9 and 500 random cases match scikit-learn to 1e-6. Edge inputs are defined: no labels gives 0.0, a ranking shorter than K is fine, repeated items count once.

## What I learned

To be written by the owner of the project.